In [ ]:
import os
import platform
import re
import platform
import warnings
import unicodedata
from datetime import datetime
from typing import List, Tuple

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from docx import Document
from striprtf.striprtf import rtf_to_text

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

MODE = 'keyword'   # 'title' or 'keyword' 

INPUT_DIR   = './Input'
OUTPUT_DIR  = './Output'

THESAURUS_XLSX = './Thesaurus/SK_Local_Thesaurus.xlsx'
THESAURUS_COL  = 'slovak_terms'
STOPWORDS_TXT  = './Input/stopword_dictionary.txt'

LLM_FALLBACK = 'nvidia/Llama-3.1-Nemotron-Nano-4B-v1.1'
LLM_PRIMARY= 'marcuscedricridia/8B-Nemotaur-IT'

MAX_NEW_TOKENS   = 1024
DO_SAMPLE        = True
TEMPERATURE      = 0.2
TOP_P            = 0.9
MAX_TEXT_CHARS   = 4000 
REPETITION_PENALTY = 1.3  

def configure_cuda_allocator():
    parts = ["max_split_size_mb:128"]
    if platform.system() == "Linux":
        parts.insert(0, "expandable_segments:True")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)

#warnings.filterwarnings("ignore", message=r".*expandable_segments not supported.*")

configure_cuda_allocator()
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")

def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            words = {line.strip().lower() for line in f if line.strip()}
        print(f'Loaded {len(words)} stopwords from {path}')
        return words
    except FileNotFoundError:
        print(f'No stopword file at {path}. Continuing without.')
        return set()
    except Exception as e:
        print(f'Failed to load stopwords: {e}')
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)

    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))

    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]

    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f'Thesaurus file not found: {xlsx_path}')
    df = pd.read_excel(xlsx_path, engine='openpyxl')
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')

    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))

    seen, terms = set(), []
    for t in parts:
        t = t.strip()
        if not t:
            continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS:
            continue
        if t not in seen:
            seen.add(t)
            terms.append(t)
    print(f'Thesaurus terms loaded: {len(terms)}')
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    """Rýchly výber termínov z tezauru, ktoré sa naozaj vyskytujú v texte."""
    hits = {}
    if not text:
        return []
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0:
            hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:max_terms]

_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

def load_primary_llm():
    qconf = BitsAndBytesConfig(
                                load_in_4bit=True,
                                bnb_4bit_compute_dtype=torch.bfloat16,
                                bnb_4bit_use_double_quant=True,
                                bnb_4bit_quant_type='nf4',
                                )
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(
                                                LLM_PRIMARY,
                                                quantization_config=qconf,
                                                device_map=device_map,
                                                dtype=torch.bfloat16,
                                                trust_remote_code=True,
                                                low_cpu_mem_usage=True,
                                                )
    tok = AutoTokenizer.from_pretrained(LLM_PRIMARY)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    pipe = pipeline(
                    "text-generation",
                    model=model,
                    tokenizer=tok,
                    device_map=device_map,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=DO_SAMPLE,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    repetition_penalty=REPETITION_PENALTY,
                    return_full_text=False,
                    )
    return pipe, f"primary:{LLM_PRIMARY}"

def load_fallback_llm():
    # Vyčisti env hacky pre fallback => úplne čistý load
    for k in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_FLASH_ATTENTION"):
        if k in os.environ:
            os.environ.pop(k)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
                                                    LLM_FALLBACK,
                                                    device_map=device_map,
                                                    dtype=dtype,
                                                    trust_remote_code=True,
                                                    low_cpu_mem_usage=True,
                                                )
    tok = AutoTokenizer.from_pretrained(LLM_FALLBACK)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    pipe = pipeline(
                    "text-generation",
                    model=model,
                    tokenizer=tok,
                    device_map=device_map,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=DO_SAMPLE,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    repetition_penalty=REPETITION_PENALTY,
                    return_full_text=False,
                    )
    return pipe, f"fallback:{LLM_FALLBACK}"

try:
    LLM_PIPE, USED_MODEL = load_primary_llm()
    print(f"[LLM] Loaded {USED_MODEL}")
except Exception as e:
    print(f"[WARN] Primary model failed: {e}")
    LLM_PIPE, USED_MODEL = load_fallback_llm()
    print(f"[LLM] Loaded {USED_MODEL}")

# Prompt setup

def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars:
        return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

def prompt_title(text: str, th_priority: List[str], th_sample: List[str]) -> str:
    pri = "\n".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = "\n".join(th_sample[:40])
    return (
        "ÚLOHA: Navrhni JEDEN presný a zrozumiteľný právny nadpis (3–12 slov) k textu nižšie.\n"
        "Požiadavky: vecný, bez bodky na konci, žiadne úvodzovky.\n"
        "Vráť len samotný nadpis, nič iné.\n\n"
        f"TEZAURUS – PRIO (zhody v texte):\n{pri}\n\n"
        f"TEZAURUS – VÝBER:\n{samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "NADPIS:"
    )

def prompt_keyword(text: str, th_priority: List[str], th_sample: List[str]) -> str:
    pri = ", ".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = ", ".join(th_sample[:40])
    return (
        "ÚLOHA: Vyber JEDEN najvhodnejší právny pojem (1–4 slová) k nasledujúcemu textu.\n"
        "Preferuj pojmy z tezauru. Vráť len samotný pojem, bez vysvetlenia a bez bodky.\n\n"
        f"TEZAURUS – PRIO: {pri}\n"
        f"TEZAURUS – VÝBER: {samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "POJEM:"
    )

def generate_once(prompt: str) -> str:
    out = LLM_PIPE(prompt)
    text = out[0]["generated_text"] if isinstance(out, list) else str(out)
    text = re.sub(r'^[\-\•\:\s]+', '', text).strip()
    text = re.sub(r'[\.，。…]+$', '', text).strip()
    text = text.strip(' "„”')
    return text.splitlines()[0].strip()


# DRIVER

def process_all(input_dir=INPUT_DIR):
    rows = []
    files = sorted(os.listdir(input_dir), key=str.lower)

    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue

        if not secs:
            print(f'No text in {f}')
            continue

        for header, text in secs:
            if not text.strip():
                continue

            short = truncate_text(text, MAX_TEXT_CHARS)
            prio_terms = terms_matched_in_text(text, max_terms=30)
            # ak nie sú zhody, vezmi „template“ sample z tezauru
            sample_terms = prio_terms if prio_terms else CANON_TERMS[:40]

            if MODE == 'title':
                prompt = prompt_title(short, prio_terms, sample_terms)
                title  = generate_once(prompt)
                rows.append({
                    "file": f,
                    "header": header,
                    "suggested_title": title,
                    "method": f"llm-only ({USED_MODEL})",
                    "priority_terms": "; ".join(prio_terms[:20])
                })
            else:
                prompt = prompt_keyword(short, prio_terms, sample_terms)
                kw     = generate_once(prompt)
                rows.append({
                    "file": f,
                    "header": header,
                    "top_keyword": kw,
                    "method": f"llm-only ({USED_MODEL})",
                    "priority_terms": "; ".join(prio_terms[:20])
                })
        print(f'Processed {f} with {len(secs)} sections.')

    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if MODE == 'title':
        df = pd.DataFrame(rows, columns=[
            "file","header","suggested_title","method","priority_terms"
        ])
        stub = "llm_only_titles"
    else:
        df = pd.DataFrame(rows, columns=[
            "file","header","top_keyword","method","priority_terms"
        ])
        stub = "llm_only_keywords"

    xlsx_path = os.path.join(OUTPUT_DIR, f"{stub}_{today}.xlsx")
    try:
        df.to_excel(xlsx_path, index=False, engine="openpyxl")
        print(f"Saved: {xlsx_path}")
    except Exception as e:
        print(f"[WARN] Excel write failed: {e}")
    return df

if __name__ == "__main__":
    df = process_all(INPUT_DIR)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))


Loaded 76 stopwords from ./Input/stopword_dictionary.txt
Thesaurus terms loaded: 3000


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


[LLM] Loaded primary:marcuscedricridia/8B-Nemotaur-IT
Processed 117694.docx with 8 sections.
Processed 117695.docx with 2 sections.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed 117696.docx with 11 sections.
Processed 117697.docx with 15 sections.
Processed 117698.docx with 8 sections.
Processed 117699.docx with 18 sections.
Processed 117700.docx with 1 sections.
Processed 117701.docx with 6 sections.
Processed 117702.docx with 12 sections.
Processed 117703.docx with 10 sections.
Saved: ./Output\llm_only_titles_2025-10-15.xlsx


,file,header,suggested_title,method,priority_terms
0,117694.docx,__DOCUMENT__,Je súdnemu konaniu o žalobe opraveného dedičky...,llm-only (primary:marcuscedricridia/8B-Nemotau...,dohoda; vyporiadanie; majetok (imanie); majeto...
1,117694.docx,po-heading-id_EHU0iUd9Y0yOqqj7VGbM1A,Podiele v dedičskom konaní aj podľa § 150 Obch...,llm-only (primary:marcuscedricridia/8B-Nemotau...,podiel; vyporiadanie; spoločník; autor
2,117694.docx,po-heading-id_c6ihO-fo_kmbMqy9OwPZ5g,Uznesenia Najvyššeho súdu Slovenskej republiky...,llm-only (primary:marcuscedricridia/8B-Nemotau...,uznesenie
3,117694.docx,po-heading-id_aWeISBCNy0mMqxGjUkaTNw,Otázka viaznosťi dohody o vyporiadani BSM v de...,llm-only (primary:marcuscedricridia/8B-Nemotau...,otázka
4,117694.docx,po-heading-id_VtH6pbc6j0qtnD6ytAcu1w,Pri vyporiadaní základných stanovísk manželov ...,llm-only (primary:marcuscedricridia/8B-Nemotau...,konanie (štádium procesného konania); majetok ...
5,117694.docx,po-heading-id_aWuBOYjuME-jlXoxHWEcHg,Preferencionalita dohody v dedičskom konaní - ...,llm-only (primary:marcuscedricridia/8B-Nemotau...,dohoda; právo; rozsudok; skutočnosti; dediči; ...
6,117694.docx,po-heading-id_-ctrw_j88Uazc2-86wvAAQ,Ustanovenie § 150 Občanského zákona jako dispo...,llm-only (primary:marcuscedricridia/8B-Nemotau...,dohoda; majetok (imanie); vyporiadanie; majeto...
7,117694.docx,po-heading-id_kizOD8s8zEyXvfzM_OhAwg,Zmerné vyporiadanie bezpodielových častí spolo...,llm-only (primary:marcuscedricridia/8B-Nemotau...,zmier; vyporiadanie; uznesenie; majetok (imani...
8,117695.docx,__DOCUMENT__,Princípy správnego rozdelování důkazního břeme...,llm-only (primary:marcuscedricridia/8B-Nemotau...,bremeno; skutočnosti; norma; dôkaz; služba (st...
9,117695.docx,po-heading-id_1OMtQ1Gs9kWEadIcZhz9ig,Zoznam výjimkov z všeobecného pravidla delenia...,llm-only (primary:marcuscedricridia/8B-Nemotau...,bremeno; skutočnosti; norma; dôkaz; služba (st...


: 